In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,RobustScaler, OneHotEncoder
from sklearn.linear_model import Ridge,Lasso,LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from scipy.stats import uniform, loguniform
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV,KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load datasets
df_train = pd.read_csv("top30_vF_xi_Hc_train_83.csv")
df_test  = pd.read_csv("top30_vF_xi_Hc_test_83.csv")

# Separate features and target
X_train = df_train.drop(columns=['critical_temp'])
y_train = df_train['critical_temp']
X_test  = df_test.drop(columns=['critical_temp'])
y_test  = df_test['critical_temp']

scaler = RobustScaler()
X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)


In [2]:
#function for evaluation
def model_evaluation(model_name,y_test,y_pred):
    print(f"{model_name}\n")
    print("Mean absolute error:",mean_absolute_error(y_test,y_pred))
    print("Mean squared error:",mean_squared_error(y_test,y_pred))
    print(f"RMSE: ",np.sqrt(mean_squared_error(y_test,y_pred)))
    print("r2_score:", r2_score(y_test,y_pred))

In [ ]:
#KNN + RandomSearchCV
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import RobustScaler
#create the pipeline
knn_pipe = Pipeline([
    ('scaler', RobustScaler()),
    ('knn', KNeighborsRegressor())
])

#parameter distribution
para_dist = {
    'knn__n_neighbors' : [5,6,7,8,9,10,11,12],
    'knn__weights' : ['uniform','distance'],
    'knn__metric' : ['euclidean','manhattan'] ,
    'knn__algorithm' : ['auto','brute','kd_tree','ball_tree'] ,
    'knn__p' : [1,2,3]
}
#cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

knn_random_search = RandomizedSearchCV(
    estimator = knn_pipe,
    param_distributions = para_dist,
    cv = cv,
    random_state = 42,
    n_iter = 50,
    n_jobs = -1,
    scoring ='r2'
)

knn_random_search.fit(X_train, y_train)

print("Best parameters:", knn_random_search.best_params_)
print("model score:", knn_random_search.best_score_)

y_pred_knn = knn_random_search.best_estimator_.predict(X_test)
model_evaluation("knn with randomsearchedCV",y_test,y_pred_knn)


In [ ]:
# svr + RandomSearchCV
svr_pipe = Pipeline([
    ('prep', RobustScaler()),
    ('svr', SVR())
])
#parameter distribution
para_dist = {
    'svr__C' : [50,100,120],
    'svr__kernel' : ['rbf'],
    'svr__gamma' : [0.01, 0.05, 0.1, 0.2],
    'svr__epsilon' : [0.006, 0.005,0.1, 0.2],
    'svr__tol' : [1e-3 , 1e-4]
}

#cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

svr_random_search = RandomizedSearchCV(
    estimator=svr_pipe,
    param_distributions=para_dist,
    cv = cv,
    random_state=42,
    n_iter=20,      
    n_jobs=-1,
    scoring='r2',
    verbose=1
)
svr_random_search.fit(X_train, y_train)

print("Best parameters:", svr_random_search.best_params_)
print("model score:",svr_random_search.best_score_)

y_pred_svr = svr_random_search.best_estimator_.predict(X_test)
model_evaluation("svr with randomizedsearchedCV",y_test,y_pred_svr)

In [ ]:
#MLP + RandomizedSearchCV
from scipy.stats import expon, loguniform

mlp_pipe = Pipeline([
    ('prep', RobustScaler()),
    ('mlp', MLPRegressor(tol=1e-4, random_state=42))
])

para_dist = {
    'mlp__hidden_layer_sizes' : [(100,50),(200,100),(300, 200, 100)],
    'mlp__activation':['tanh'],
    'mlp__learning_rate': ['constant', 'adaptive'],
    'mlp__solver': ['adam'],#,'lbfgs'],
    'mlp__batch_size' : [100,256],
    'mlp__alpha':[0.0005,0.0006, 0.0007]
}
from scipy.stats import loguniform, uniform

cv = KFold(n_splits=5, shuffle=True, random_state=42)

mlp_random_search = RandomizedSearchCV(
    estimator=mlp_pipe,
    param_distributions=para_dist,
    n_iter=20,
    cv = cv,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=2,
)

mlp_random_search.fit(X_train, y_train)

print("Best parameters:", mlp_random_search.best_params_)
print("model score:", mlp_random_search.best_score_)

y_pred_mlp = mlp_random_search.best_estimator_.predict(X_test)
model_evaluation("mlp with randomizedsearchedCV",y_test,y_pred_mlp)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


# 1. Build the Pipeline
# This ensures data is scaled before hitting the Random Forest
rf_pipeline = Pipeline([
    ('prep', RobustScaler()),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1))
])

# 2. Define a much more aggressive parameter grid
# Notice we use 'rf__' to tell the pipeline these parameters belong to the model
rf_params = {
    'rf__n_estimators': [2000],        # More trees for stability
    'rf__max_depth': [30],            # Allow deeper trees to capture complexity
    'rf__min_samples_split': [3]          # Fine-tuning the splits
}

rf_search = RandomizedSearchCV(
    rf_pipeline, 
    param_distributions=rf_params, 
    n_iter=30, 
    cv=5, 
    scoring='r2', 
    n_jobs=-1
)

# 3. Fit the model
rf_search.fit(X_train, y_train)

# 4. Extract results
best_pipeline = rf_search.best_estimator_
print("Best parameters:", rf_search.best_params_)

# 5. Test it out
y_pred = best_pipeline.predict(X_test)

print("model score:", rf_search.best_score_)

y_pred_rf = rf_search.best_estimator_.predict(X_test)
model_evaluation("random forest",y_test,y_pred_rf)

In [ ]:
from lightgbm import LGBMRegressor

lgb_pipe = Pipeline(steps=[
    ('prep', RobustScaler()),
    ('lgb', LGBMRegressor(random_state=42, verbose=-1, n_jobs=-1))
])

lgb_params = {
    'lgb__n_estimators': [800, 1000, 1200, 1500],
    'lgb__learning_rate': [0.003,0.004,0.005, 0.01,0.05],
    'lgb__num_leaves': [40, 50, 100, 150, 200,250],
    'lgb__max_depth': [4,5,6, 8, -1],
    'lgb__min_child_samples': [10, 20, 30, 50],
    'lgb__subsample':[0.7, 0.8, 0.9, 1.0],
    'lgb__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'lgb__reg_alpha': [0.0, 0.1, 0.5, 1.0],
    'lgb__reg_lambda': [0.0, 0.1, 0.5, 1.0]
}

lgb_search = RandomizedSearchCV(
    estimator=lgb_pipe, 
    param_distributions=lgb_params, 
    n_iter=30, 
    cv=5, 
    scoring='r2', 
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# print("Training LightGBM...")
lgb_search.fit(X_train, y_train)

print("Best parameters:", lgb_search.best_params_)
print("model score:", lgb_search.best_score_)

y_pred_lgb = lgb_search.best_estimator_.predict(X_test)
model_evaluation("lgb with randomizedsearchedCV",y_test,y_pred_lgb)

In [ ]:
import joblib

knn_best = knn_random_search.best_estimator_
svr_best = svr_random_search.best_estimator_
mlp_best = mlp_random_search.best_estimator_
best_rf = rf_search.best_estimator_
best_lgb = lgb_search.best_estimator_


# #saving in model folder as .pkl for re-useability
joblib.dump(knn_best , "models_83/tuning_50_HPO/knn_best.pkl")
joblib.dump(svr_best , "models_83/tuning_50_HPO/svr_best.pkl")
joblib.dump(mlp_best, "models_83/tuning_50_HPO/mlp_best.pkl")
joblib.dump(best_rf , "models_83/tuning_50_HPO/best_rf.pkl")
joblib.dump(best_lgb, "models_83/tuning_50_HPO/best_lgb.pkl")

In [ ]:
#loading the input files
import joblib 

knn_best = joblib.load("models_83/tuning_50_HPO/knn_best.pkl")
svr_best = joblib.load("models_83/tuning_50_HPO/svr_best.pkl")
mlp_best = joblib.load("models_83/tuning_50_HPO/mlp_best.pkl")
best_rf = joblib.load("models_83/tuning_50_HPO/best_rf.pkl")
best_lgb = joblib.load("models_83/tuning_50_HPO/best_lgb.pkl")

base_model_83= [
    ('knn', knn_best),
    ('svr' , svr_best),
    ('mlp', mlp_best),
    ('lgbm', best_lgb),
    ('rf', best_rf )
]


In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV
# 2. Define the simple linear meta model
# This acts as a smart weighted average to prevent overfitting

stack_ridge = StackingRegressor(
    estimators= base_model_83 ,
    final_estimator=Ridge(alpha=2.0, random_state=42),
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    passthrough=False, # Keep it clean
    n_jobs=-1
)

# 4. Train and Evaluate the Model
print("Training the Stacking Ensemble...")
stack_ridge.fit(X_train_scale, y_train)

# --- 3. Calculate Train R2 Score ---
y_pred_train = stack_ridge.predict(X_train_scale)
train_r2 = r2_score(y_train, y_pred_train)

print(f"Train R2 Score: {train_r2:.4f}")
print("Generating predictions...")
y_pred = stack_ridge.predict(X_test_scale)
print("Mean absolute error:",mean_absolute_error(y_test,y_pred))
print("Mean squared error:",mean_squared_error(y_test,y_pred))
print(f"RMSE: ",np.sqrt(mean_squared_error(y_test,y_pred)))
print("r2_score:", r2_score(y_test,y_pred))

In [ ]:
import joblib

# ── Save trained stacking model ────────────────────────
joblib.dump(stack_ridge, "stacking_model_30features.pkl")

# ── Save RobustScaler ─────────────────────────────────
joblib.dump(scaler, "robust_scaler.pkl")

print("Model saved: stacking_model_30features.pkl")
print("Scaler saved: robust_scaler.pkl")

# ── Verify they load correctly ───────────────────────────────
model_check  = joblib.load("stacking_model_30features.pkl")
scaler_check = joblib.load("robust_scaler.pkl")
print("Verification: both files load correctly ✅")

In [ ]:
# Predicted vs Actual Plot
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--')
plt.xlabel("Actual $T_c$", fontsize=12)
plt.ylabel("Predicted $T_c$", fontsize=12)
# plt.title("Predicted vs Actual $T_c$ (Stacking Model)", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
#save high resolution image
plt.savefig("predicted_vs_actual_Tc.png", dpi =1000, bbox_inches = 'tight')
plt.show()